In [1]:
# Load packages
import os

# Set workspace environment variables
os.environ["WORKSPACE_BUCKET"] = "gs://rw-migration-aou-rw-9f879fc4"
os.environ["ARTIFACT_REGISTRY_DOCKER_REPO"] = (
    "us-central1-docker.pkg.dev/all-of-us-rw-prod/aou-rw-gar-remote-repo-docker-prod"
)

# Load the remaining workspace environment settings
%run Setting_Env_Variables.ipynb

Successfully set WORKSPACE_CDR to: wb-silky-artichoke-2408.C2024Q3R9
Found bucket: id=dataproc-staging-aou-rw-9f879fc4, bucketName=dataproc-staging-wb-fast-radish-6415
Found bucket: id=dataproc-temp-aou-rw-9f879fc4, bucketName=dataproc-temp-wb-fast-radish-6415
Found bucket: id=rw-migration-aou-rw-9f879fc4, bucketName=rw-migration-aou-rw-9f879fc4
Found bucket: id=vwb-aou-datasets-controlled-v8r9, bucketName=vwb-aou-datasets-controlled

Variables extracted:
GOOGLE_CLOUD_PROJECT: wb-fast-radish-6415
WORKSPACE_BUCKET: gs://rw-migration-aou-rw-9f879fc4
WORKSPACE_TEMP_BUCKET: NOT FOUND
WORKSPACE_CDR: wb-silky-artichoke-2408.C2024Q3R9

✅ Saved to /home/dataproc/.bashrc
C2024Q3R9 BQ_DATASET
JupyterLab_Spark_cluster_for_AoU_Spark_20261024 DATAPROC_CLUSTER
dataproc-staging-aou-rw-9f879fc4 GCS_BUCKET
dataproc-temp-aou-rw-9f879fc4 GCS_BUCKET
rw-migration-aou-rw-9f879fc4 GCS_BUCKET
vwb-aou-datasets-controlled-v8r9 GCS_BUCKET
GOOGLE_CLOUD_PROJECT = wb-fast-radish-6415
WORKSPACE_BUCKET = gs://rw-migr

In [79]:
%%writefile update.sh

plink2 \
    --bfile "${GENO}/EUR_ab0.35_dp20_call0.9_hwe1e15_chr${CHROM}" \
    --set-all-var-ids 'chr@:#:$r:$a' \
    --new-id-max-allele-len 100 \
    --memory 3100 \
    --make-bed \
    --out "${OUTPUT}/EUR_ab0.35_dp20_call0.9_hwe1e15_updated_chr${CHROM}"

Overwriting update.sh


In [82]:
# Logging directory
LOGGING=f"{os.environ['WORKSPACE_BUCKET']}/data/logging"
# Output directory 
OUTPUT=f"{os.environ['WORKSPACE_BUCKET']}/data/chip/plink"
# Path to genomic data 
MNT_PATH="/mnt/data/input/gs/rw-migration-aou-rw-9f879fc4/data/chip/plink"

DISK_SIZE = 4 
    
# Loop to submit dsub command 
for CHROM in [2]:    
    
    print(f"Submitting dsub command for chromosome {CHROM}")

    BED=f'gs://rw-migration-aou-rw-9f879fc4/data/chip/plink/EUR_ab0.35_dp20_call0.9_hwe1e15_chr{CHROM}.bed'
    BIM=f'gs://rw-migration-aou-rw-9f879fc4/data/chip/plink/EUR_ab0.35_dp20_call0.9_hwe1e15_chr{CHROM}.bim'
    FAM=f'gs://rw-migration-aou-rw-9f879fc4/data/chip/plink/EUR_ab0.35_dp20_call0.9_hwe1e15_chr{CHROM}.fam'
    
    # DSUB command 
    command = [
        "aou_dsub",
        "--project", os.getenv('GOOGLE_CLOUD_PROJECT'), 
        "--user-project", os.getenv('GOOGLE_CLOUD_PROJECT'), 
        "--image", f"{os.environ['ARTIFACT_REGISTRY_DOCKER_REPO']}/gfanz/plink2:latest",
        "--disk-size", str(DISK_SIZE),
        "--machine-type", "n1-standard-1",
        "--logging", LOGGING,
        "--input", f"BED={BED}",
        "--input", f"BIM={BIM}",
        "--input", f"FAM={FAM}",
        "--output-recursive", f"OUTPUT={OUTPUT}",
        "--env", f"GENO={MNT_PATH}",
        "--env", f"CHROM={CHROM}",
        "--script", "update.sh"
    ]

    # Run in same shell as source command
    dsub_command = subprocess.run(
        f"source ~/aou_dsub.bash && {' '.join(command)}",
        shell=True,
        executable="/bin/bash",
        capture_output=True,
        text=True
    )

    print(dsub_command.stdout)
    print(dsub_command.stderr)

Submitting dsub command for chromosome 2
update--vienose--261025-140316-33

Job properties:
  job-id: update--vienose--261025-140316-33
  job-name: update
  user-id: vienose
Provider internal-id (operation): projects/wb-fast-radish-6415/locations/us-central1/jobs/update--vienose--261025-140316-33-0-0
Launched job-id: update--vienose--261025-140316-33
To check the status, run:
  dstat --provider google-batch --project wb-fast-radish-6415 --location us-central1 --jobs 'update--vienose--261025-140316-33' --users 'vienose' --status '*'
To cancel the job, run:
  ddel --provider google-batch --project wb-fast-radish-6415 --location us-central1 --jobs 'update--vienose--261025-140316-33' --users 'vienose'

